In [11]:
import os
os.chdir("/Users/caiwansun/Downloads/timeline-example-caiwan")
print(os.getcwd())

/Users/caiwansun/Downloads/timeline-example-caiwan


In [15]:
import json
import os
import time
from datetime import timedelta


def time_to_seconds(time_str):
    if ":" in time_str:
        h, m, s = time_str.split(":")
        return int(h) * 3600 + int(m) * 60 + float(s)
    else:
        return float(time_str)


def seconds_to_tc(seconds):
    td = timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    secs = total_seconds % 60
    return f"{hours:02}:{minutes:02}:{secs:02}.0000"


def clean_segments(segments):
    processed = []

    for seg in segments:
        start = time_to_seconds(seg["time_in"])
        end = time_to_seconds(seg["time_out"])

        if end > start:
            processed.append((start, end))

    processed.sort(key=lambda x: x[0])

    merged = []

    for start, end in processed:
        if not merged:
            merged.append([start, end])
        else:
            last_start, last_end = merged[-1]

            if start <= last_end:
                merged[-1][1] = max(last_end, end)
            else:
                merged.append([start, end])

    return merged

def convert_llm_to_amalia(input_file, output_file, metadata_id, label):

    with open(input_file, "r") as f:
        data = json.load(f)

    raw_segments = data["segments"]
    cleaned = clean_segments(raw_segments)

    # 构建 level1 列表
    level1_segments = []

    for start, end in cleaned:
        level1_segments.append({
            "tclevel": 1,
            "tcin": seconds_to_tc(start),
            "tcout": seconds_to_tc(end),
            "label": label
        })

    amalia_format = {
        "type": "text",
        "id": metadata_id,
        "algorithm": "LLM",
        "processor": "ChatGPT",
        "processed": int(time.time() * 1000),
        "version": 1,
        "localisation": [
            {
                "sublocalisations": {
                    "localisation": level1_segments
                },
                "type": "text",
                "tcin": "00:00:00.0000",
                "tcout": "00:10:27.0000",
                "tclevel": 0
            }
        ]
    }

    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, "w") as f:
        json.dump(amalia_format, f, indent=4)

    print("✅ Generated:", output_file)
    print("Segments after merge:", len(cleaned))

if __name__ == "__main__":

    convert_llm_to_amalia(
        "gpt4.json",
        "samples-data/data-final/d2g2-segments-messi-gpt4-enact.json",
        "d2g2-segments-messi-gpt4-enact",
        "ENACTING MESSI"
    )

    convert_llm_to_amalia(
        "gpt52.json",
        "samples-data/data-final/d2g2-segments-messi-gpt52-enact.json",
        "d2g2-segments-messi-gpt52-enact",
        "ENACTING MESSI"
    )

✅ Generated: samples-data/data-final/d2g2-segments-messi-gpt4-enact.json
Segments after merge: 4
✅ Generated: samples-data/data-final/d2g2-segments-messi-gpt52-enact.json
Segments after merge: 6
